In [1]:
from datetime import datetime
import pandas as pd
from meteostat import Stations, Monthly

# 1) Time span
start = datetime(2013, 1, 1)
end   = datetime(2026, 1, 1)

# 2) Spain stations with data in window (daily inventory = proxy for monthly availability)
st = Stations().region('ES')
st = st.inventory('daily', (start, end))           # optional but recommended
stations_df = st.fetch()
stations_df_fil = stations_df[~stations_df['region'].isin(['IB', 'CN'])]
# stations_df_fil = stations_df.copy()
# assert not stations_df_fil.empty, "No ES stations matched; relax filters or shorten the date range."

# 3) Use station IDs (list) for Monthly — avoids DataFrame edge cases across versions
station_ids = stations_df_fil.index.astype(str).tolist()

# 4) Fetch monthly data
m = Monthly(station_ids, start=start, end=end)      # model=True by default
df = m.fetch()                                      # columns: station, time, tavg, tmin, tmax, etc.

df


tavg  tmin  tmax   prcp  wspd    pres     tsun
station time                                                      
08001   2013-01-01  11.4   8.6  14.2  205.0  <NA>  1019.7   5460.0
        2013-02-01  10.7   8.1  13.3   90.0  <NA>  1020.5   4860.0
        2013-03-01  11.9   9.1  14.8  216.0  <NA>  1004.7   7560.0
        2013-04-01  13.0   9.8  16.1   92.0  <NA>  1016.7  12540.0
        2013-05-01  13.2  10.2  16.2   73.0  <NA>  1018.8  13680.0
...                  ...   ...   ...    ...   ...     ...      ...
60338   2025-09-01  24.5  21.2  27.9    0.5  <NA>  1017.0  15360.0
        2025-10-01  22.8  19.6  26.0    2.0  <NA>  1017.2  13980.0
        2025-11-01  17.8  14.1  21.4   47.0  <NA>  1018.0  11520.0
        2025-12-01  14.5  11.1  17.9  100.0  <NA>  1016.9   8640.0
        2026-01-01  14.3  11.9  17.3   84.2  <NA>  1015.4   7280.0

[9495 rows x 7 columns]

In [10]:
# Fetch monthly data for excluded regions (Balearic Islands and Canary Islands)
# ibcan_ids = stations_df[stations_df['region'].isin(['IB', 'CN'])].index.astype(str).tolist()
df_ibcan = Monthly(stations_df, start=start, end=end).fetch()

# Merge on index (union of both MultiIndex keys); df takes priority where indices overlap
df_combined = pd.merge(df, df_ibcan, how='outer', left_index=True, right_index=True, suffixes=('', '_ibcan'))
df_combined


tavg  tmin  tmax   prcp  wspd    pres     tsun  \
station time                                                         
08001   2013-01-01  11.4   8.6  14.2  205.0  <NA>  1019.7   5460.0   
        2013-02-01  10.7   8.1  13.3   90.0  <NA>  1020.5   4860.0   
        2013-03-01  11.9   9.1  14.8  216.0  <NA>  1004.7   7560.0   
        2013-04-01  13.0   9.8  16.1   92.0  <NA>  1016.7  12540.0   
        2013-05-01  13.2  10.2  16.2   73.0  <NA>  1018.8  13680.0   
...                  ...   ...   ...    ...   ...     ...      ...   
60338   2025-09-01  24.5  21.2  27.9    0.5  <NA>  1017.0  15360.0   
        2025-10-01  22.8  19.6  26.0    2.0  <NA>  1017.2  13980.0   
        2025-11-01  17.8  14.1  21.4   47.0  <NA>  1018.0  11520.0   
        2025-12-01  14.5  11.1  17.9  100.0  <NA>  1016.9   8640.0   
        2026-01-01  14.3  11.9  17.3   84.2  <NA>  1015.4   7280.0   

                    tavg_ibcan  tmin_ibcan  tmax_ibcan  prcp_ibcan  \
station time                                                         
08001   2013-01-01        11.4         8.6        14.2       205.0   
        2013-02-01        10.7         8.1        13.3        90.0   
        2013-03-01        11.9         9.1        14.8       216.0   
        2013-04-01        13.0         9.8        16.1        92.0   
        2013-05-01        13.2        10.2        16.2        73.0   
...                        ...         ...         ...         ...   
60338   2025-09-01        24.5        21.2        27.9         0.5   
        2025-10-01        22.8        19.6        26.0         2.0   
        2025-11-01        17.8        14.1        21.4        47.0   
        2025-12-01        14.5        11.1        17.9       100.0   
        2026-01-01        14.3        11.9        17.3        84.2   

                    wspd_ibcan  pres_ibcan  tsun_ibcan  
station time                                            
08001   2013-01-01        <NA>      1019.7      5460.0  
        2013-02-01        <NA>      1020.5      4860.0  
        2013-03-01        <NA>      1004.7      7560.0  
        2013-04-01        <NA>      1016.7     12540.0  
        2013-05-01        <NA>      1018.8     13680.0  
...                        ...         ...         ...  
60338   2025-09-01        <NA>      1017.0     15360.0  
        2025-10-01        <NA>      1017.2     13980.0  
        2025-11-01        <NA>      1018.0     11520.0  
        2025-12-01        <NA>      1016.9      8640.0  
        2026-01-01        <NA>      1015.4      7280.0  

[11065 rows x 14 columns]

In [11]:
import os

# Aggregate all numeric columns; prcp is summed, everything else is averaged
sum_cols = {'prcp'}
agg_funcs = {
    col: 'sum' if col in sum_cols else 'mean'
    for col in df_combined.select_dtypes('number').columns
}

df_monthly = (
    df_combined
      .groupby(level='time')   # time is a MultiIndex level, not a column
      .agg(agg_funcs)
      .round(2)
)

os.makedirs(os.path.abspath('outputs'), exist_ok=True)
output_path = os.path.join(os.path.abspath('outputs'), 'spain_monthly_avg_temperature.xlsx')
df_monthly.to_excel(output_path, index=True)
df_monthly

,tavg,tmin,tmax,prcp,wspd,pres,tsun,tavg_ibcan,tmin_ibcan,tmax_ibcan,prcp_ibcan,wspd_ibcan,pres_ibcan,tsun_ibcan
time,,,,,,,,,,,,,,
2013-01-01,8.74,4.39,12.96,4997.0,<NA>,1020.79,8331.27,9.7,5.48,13.78,73.21,<NA>,1021.06,8897.54
2013-02-01,8.1,3.72,12.46,4141.0,<NA>,1018.72,8808.21,9.06,4.85,13.27,61.28,<NA>,1018.68,9023.64
2013-03-01,10.95,6.83,15.08,8250.0,<NA>,1007.77,9057.86,11.79,7.74,15.85,121.66,<NA>,1008.49,9754.55
2013-04-01,13.01,7.85,18.16,3906.0,<NA>,1015.28,13456.36,13.73,8.78,18.67,57.32,<NA>,1015.32,13511.25
2013-05-01,14.69,9.3,20.09,2808.0,<NA>,1015.59,15590.36,15.24,10.11,20.38,40.58,<NA>,1015.89,15405.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-01,21.15,14.87,27.44,1791.5,<NA>,1017.35,15403.45,21.47,15.56,27.38,26.92,<NA>,1017.43,15488.82
2025-10-01,18.32,12.8,23.83,3136.0,<NA>,1017.4,12753.1,18.73,13.52,23.94,46.13,<NA>,1017.47,12863.82
2025-11-01,12.0,7.43,16.58,5538.0,<NA>,1017.37,9409.66,12.94,8.54,17.35,84.37,<NA>,1017.46,9785.29
